# 01 · Fetch raw data

**Paper:** *Racing With the Machine: AI Exposure, Skill, and the Future of the Professions in the Second Machine Age*
Sarthak Pattnaik, Eugene Pinsky, Kathleen Park — Boston University. Target journal: *Merits* (MDPI).

This notebook downloads every raw input once and caches it under `data/raw/`. Re-running is cheap: cached files are not downloaded again (delete a file to force a refresh).

| Source | Key | Feeds |
|---|---|---|
| BLS OEWS national files, May 2019–2025 | none | H2, H3, H4 |
| O*NET database (Job Zones, Skills, Work Activities) | none | H1, H3 |
| Eloundou et al. (2024) LLM exposure, `occ_level.csv` | none | H1–H4 |
| BEA GDP by Industry (components of value added) | BEA | H5 |
| BEA Fixed Assets (intellectual property investment by industry) | BEA | H5 |
| Census Business Trends and Outlook Survey (AI use) | none | H5 robustness |
| FRED (labour share, productivity, graduate unemployment) | FRED | context figure |

**Run order:** 01 → 02 → 03 → 04 → 05 → 06. Each notebook prints a short check at the end; send me that output if anything looks off.

In [1]:
# If a package is missing, uncomment and run once:
# %pip install pandas numpy requests openpyxl statsmodels matplotlib jinja2 scipy
from rwm_common import *
import zipfile, io, re
env = load_env()
BEA_KEY = get_key(env, "BEA_API_KEY", "BEA_KEY", "BEA")
FRED_KEY = get_key(env, "FRED_API_KEY", "FRED_KEY", "FRED")
status = {}   # source -> short status string, printed at the end

.env found at C:\Users\91838\Downloads\Recent\.env with keys: ['BEA_API_KEY', 'CENSUS_API_KEY', 'FRED_API_KEY']


## 1. BLS OEWS national files

One zip per May reference year. BLS has used two folder spellings (`special-requests`, `special.requests`); both are tried. Years not yet published are skipped, not failed.

In [2]:
oews_zips = {}
for y in OEWS_YEARS:
    yy = str(y)[2:]
    urls = [f"https://www.bls.gov/oes/special-requests/oesm{yy}nat.zip",
            f"https://www.bls.gov/oes/special.requests/oesm{yy}nat.zip"]
    try:
        path, used = download_first(urls, DATA_RAW / "oews" / f"oesm{yy}nat.zip", retries=2)
        oews_zips[y] = path
        print(f"{y}: OK  ({used})")
    except Exception as e:
        print(f"{y}: not available -> {str(e).splitlines()[1] if len(str(e).splitlines())>1 else e}")
status["OEWS years"] = sorted(oews_zips)

2019: OK  (https://www.bls.gov/oes/special-requests/oesm19nat.zip)
2020: OK  (https://www.bls.gov/oes/special-requests/oesm20nat.zip)
2021: OK  (https://www.bls.gov/oes/special-requests/oesm21nat.zip)
2022: OK  (https://www.bls.gov/oes/special-requests/oesm22nat.zip)
2023: OK  (https://www.bls.gov/oes/special-requests/oesm23nat.zip)
2024: OK  (https://www.bls.gov/oes/special-requests/oesm24nat.zip)
2025: OK  (https://www.bls.gov/oes/special-requests/oesm25nat.zip)


In [3]:
WAGE_COLS = ["a_mean", "a_pct10", "a_pct25", "a_median", "a_pct75", "a_pct90"]

def read_oews(path: Path, year: int) -> pd.DataFrame:
    """Read the national cross-industry file inside an OEWS zip; keep detailed occupations."""
    with zipfile.ZipFile(path) as z:
        names = [n for n in z.namelist() if n.lower().endswith((".xlsx", ".xls"))
                 and "field" not in n.lower() and "description" not in n.lower()]
        pref = [n for n in names if re.search(r"national", n, re.I)]
        member = (pref or names)[0]
        df = pd.read_excel(io.BytesIO(z.read(member)), dtype=str)
    df.columns = [str(c).strip().lower() for c in df.columns]
    grp = next((c for c in ("o_group", "occ_group") if c in df.columns), None)
    if grp is None:
        raise ValueError(f"{year}: no occupation-group column in {member}: {list(df.columns)[:15]}")
    df = df[df[grp].str.strip().str.lower() == "detailed"].copy()
    keep = ["occ_code", "occ_title", "tot_emp", *WAGE_COLS, "annual"]
    df = df[[c for c in keep if c in df.columns]]
    for c in WAGE_COLS:
        raw = df[c].astype(str).str.strip()
        df[c + "_topcoded"] = raw.eq("#")      # '#' = wage at or above the OEWS top code
        df[c] = to_num(raw)                     # '*', '**', '#' become NaN
    df["tot_emp"] = to_num(df["tot_emp"])
    df["year"] = year
    df["source_file"] = member
    return df

oews = pd.concat([read_oews(p, y) for y, p in oews_zips.items()], ignore_index=True)
oews.to_csv(DATA_INT / "oews_national_long.csv", index=False)
print(oews.groupby("year").agg(occupations=("occ_code", "nunique"),
                               employment_m=("tot_emp", lambda s: s.sum() / 1e6),
                               p90_topcoded=("a_pct90_topcoded", "sum")).round(2))

      occupations  employment_m  p90_topcoded
year                                         
2019          789        146.87            33
2020          789        139.10            38
2021          831        140.89            51
2022          830        147.88            39
2023          831        151.85            41
2024          831        154.19            45
2025          830        155.48             0


## 2. O*NET database

The newest release available is used. Only four tab-separated files are needed.

In [4]:
onet_zip, onet_version = None, None
for v in ONET_VERSIONS:
    try:
        onet_zip, _ = download_first([f"https://www.onetcenter.org/dl_files/database/db_{v}_text.zip"],
                                     DATA_RAW / "onet" / f"db_{v}_text.zip", retries=1)
        onet_version = v
        break
    except Exception:
        print(f"O*NET {v}: not found")
if onet_zip is None:
    raise RuntimeError("No O*NET release downloaded; check https://www.onetcenter.org/database.html")
print("O*NET release:", onet_version)

ONET_FILES = {"job_zones": "Job Zones.txt", "skills": "Skills.txt",
              "work_activities": "Work Activities.txt", "occupations": "Occupation Data.txt"}
with zipfile.ZipFile(onet_zip) as z:
    for short, fname in ONET_FILES.items():
        member = next(n for n in z.namelist() if n.endswith("/" + fname) or n == fname)
        df = pd.read_csv(io.BytesIO(z.read(member)), sep="\t", dtype=str)
        df.to_csv(DATA_INT / f"onet_{short}.csv", index=False)
        print(f"{short:16s} rows={len(df):>7,}")
status["O*NET"] = onet_version

O*NET release: 30_1
job_zones        rows=    923
skills           rows= 62,580
work_activities  rows= 73,308
occupations      rows=  1,016


## 3. LLM exposure scores (Eloundou, Manning, Mishkin & Rock, *Science* 2024)

`dv_rating_*` are GPT-4 ratings, `human_rating_*` are annotator ratings. α = direct exposure (E1), β = E1 + 0.5·E2, γ = E1 + E2, where E2 is exposure that needs extra software built on the LLM.

In [5]:
p = download(EXPOSURE_URL, DATA_RAW / "exposure" / "occ_level.csv")
expo = pd.read_csv(p)
print(expo.shape); print(expo.head(3))
status["Exposure"] = f"{len(expo)} O*NET occupations"

(923, 8)
  O*NET-SOC Code                            Title  dv_rating_alpha  \
0     11-1011.00                 Chief Executives         0.100000   
1     11-1011.03    Chief Sustainability Officers         0.166667   
2     11-1021.00  General and Operations Managers         0.000000   

   dv_rating_beta  dv_rating_gamma  human_rating_alpha  human_rating_beta  \
0        0.460000         0.820000            0.180000           0.350000   
1        0.555556         0.944444            0.055556           0.388889   
2        0.480769         0.961538            0.115385           0.384615   

   human_rating_gamma  
0            0.520000  
1            0.722222  
2            0.653846  


## 4. BEA: components of value added by industry

Table IDs are looked up by name rather than hard-coded, because BEA renumbers tables after comprehensive revisions.

In [6]:
tids = bea_param_values(BEA_KEY, "GDPbyIndustry", "TableID")
kcol = next(c for c in tids.columns if c.lower() == "key")
dcol = next(c for c in tids.columns if c.lower().startswith("desc"))
cand = tids[tids[dcol].str.contains("Components of Value Added by Industry", case=False)
            & ~tids[dcol].str.contains("percent", case=False)]
print(cand.to_string(index=False))
TABLE_COMPONENTS = str(cand[kcol].iloc[0])

va_raw = bea_get(BEA_KEY, f"bea_gdpbyind_table{TABLE_COMPONENTS}", DatasetName="GDPbyIndustry",
                 TableID=TABLE_COMPONENTS, Frequency="A", Year="ALL", Industry="ALL")
va_raw.to_csv(DATA_INT / "bea_components_raw.csv", index=False)
print(va_raw.shape); print(va_raw.head())
status["BEA value added"] = f"table {TABLE_COMPONENTS}, {va_raw['Year'].nunique()} years"

Key                                      Desc
  6 Components of Value Added by Industry (A)
(11484, 8)
  TableID Frequency  Year Quarter Industry  \
0       6         A  1997    1997       11   
1       6         A  1997    1997       11   
2       6         A  1997    1997       11   
3       6         A  1997    1997       11   
4       6         A  1997    1997    111CA   

                              IndustrYDescription DataValue NoteRef  
0  Taxes on production and imports less subsidies      -1.1       6  
1                         Gross operating surplus      83.1       6  
2     Agriculture, forestry, fishing, and hunting     108.6       6  
3                       Compensation of employees      26.6       6  
4                       Compensation of employees      17.1       6  


## 5. BEA: investment in intellectual property products by industry

Intellectual property products = software, R&D, and entertainment originals. Used to build each industry's pre-2008 IP intensity.

In [7]:
tn = bea_param_values(BEA_KEY, "FixedAssets", "TableName")
ncol = next(c for c in tn.columns if c.lower() == "tablename")
dcol = next(c for c in tn.columns if c.lower().startswith("desc"))
ipc = tn[tn[dcol].str.contains("Intellectual Property Products by Industry", case=False)
         & tn[dcol].str.contains("Investment", case=False)
         & ~tn[dcol].str.contains("Chain|Quantity|Price|Contribution", case=False, regex=True)]
print(ipc.to_string(index=False))
IP_TABLE = str(ipc[ncol].iloc[0])

ip_raw = bea_get(BEA_KEY, f"bea_fixedassets_{IP_TABLE}", DatasetName="FixedAssets",
                 TableName=IP_TABLE, Year="ALL")
ip_raw.to_csv(DATA_INT / "bea_ip_investment_raw.csv", index=False)
print(ip_raw.shape); print(ip_raw.head())
status["BEA IP investment"] = IP_TABLE

TableName                                                                      Description
 FAAt307I Table 3.7I. Investment in Private Intellectual Property Products by Industry (A)
(7584, 10)
  TableName    SeriesCode LineNumber       LineDescription TimePeriod  \
0  FAAt307I  i3ptotl1ip00          1  Private fixed assets       1947   
1  FAAt307I  i3ptotl1ip00          1  Private fixed assets       1948   
2  FAAt307I  i3ptotl1ip00          1  Private fixed assets       1949   
3  FAAt307I  i3ptotl1ip00          1  Private fixed assets       1950   
4  FAAt307I  i3ptotl1ip00          1  Private fixed assets       1951   

       METRIC_NAME CL_UNIT UNIT_MULT DataValue   NoteRef  
0  Historical Cost   Level         6     2,020  FAAt307I  
1  Historical Cost   Level         6     2,075  FAAt307I  
2  Historical Cost   Level         6     2,035  FAAt307I  
3  Historical Cost   Level         6     2,256  FAAt307I  
4  Historical Cost   Level         6     2,354  FAAt307I  


## 6. Census Business Trends and Outlook Survey (AI use)

Optional input. If the sector file's layout has changed, the rest of the pipeline still runs; notebook 05 skips the AI-use check.

In [8]:
for k, url in BTOS_URLS.items():
    try:
        download(url, DATA_RAW / "btos" / f"{k}.xlsx", retries=2)
        print(f"BTOS {k}: OK")
        status[f"BTOS {k}"] = "OK"
    except Exception as e:
        print(f"BTOS {k}: not downloaded ({e})")
        status[f"BTOS {k}"] = "missing (optional)"

BTOS national: OK
BTOS sector: OK


## 7. FRED context series

In [9]:
fred = []
for sid, label in FRED_SERIES.items():
    try:
        d = fred_series(FRED_KEY, sid); d["label"] = label; fred.append(d)
        print(f"{sid:12s} {d['date'].min():%Y-%m} to {d['date'].max():%Y-%m}")
    except Exception as e:
        print(f"{sid}: skipped ({e})")
pd.concat(fred).to_csv(DATA_INT / "fred.csv", index=False)
status["FRED series"] = len(fred)

PRS85006173  1947-01 to 2026-04
OPHNFB       1947-01 to 2026-04
CGBD2024     2000-01 to 2026-09
LNS14027662  1992-01 to 2026-09


## Check

In [10]:
for k, v in status.items():
    print(f"{k:22s} {v}")

OEWS years             [2019, 2020, 2021, 2022, 2023, 2024, 2025]
O*NET                  30_1
Exposure               923 O*NET occupations
BEA value added        table 6, 29 years
BEA IP investment      FAAt307I
BTOS national          OK
BTOS sector            OK
FRED series            4
